In [ ]:
file_path = "/content/"

In [ ]:
import networkx as nx
import pandas as pd
from collections import Counter
from collections import defaultdict
from google.colab import data_table
from google.colab import files
import requests
import re
from tqdm import tqdm
import time

#Leitura do arquivo de rede
with open(file_path, 'r', encoding='utf-8') as f:
    lines = f.readlines()

#Etapas de leitura e parsing dos blocos
vertex_labels = []
edges = []
cluster_names = []
abstracts = []
titles = []
wos_ids = []
cluster_indices = []

mode = None

In [ ]:
#Identificação dos blocos
for line in lines:
    stripped = line.strip()
    if stripped.startswith("#vertices"):
        mode = "vertices"
        continue
    elif stripped.startswith("#edges"):
        mode = "edges"
        continue
    elif stripped.startswith('#v "ClusterIndex"'):
        mode = "cluster_index"
        continue
    elif stripped.startswith('#v "ClusterName"'):
        mode = "cluster_name"
        continue
    elif stripped.startswith('#v "abstract"'):
        mode = "abstract"
        continue
    elif stripped.startswith('#v "title"'):
        mode = "title"
        continue
    elif stripped.startswith('#v "wos_id"'):
        mode = "wos_id"
        continue

    if mode == "vertices":
        vertex_labels.append(stripped.strip('"'))
    elif mode == "edges":
        parts = stripped.split()
        if len(parts) == 2:
            u, v = map(int, parts)
            edges.append((u, v))
        elif len(parts) == 3:
            u, v = map(int, parts[:2])
            weight = float(parts[2])
            edges.append((u, v, weight))
    elif mode == "cluster_index":
        cluster_indices.append(int(stripped))
    elif mode == "cluster_name":
        cluster_names.append(stripped.strip('"'))
    elif mode == "abstract":
        abstracts.append(stripped.strip('"'))
    elif mode == "title":
        titles.append(stripped.strip('"'))
    elif mode == "wos_id":
        wos_ids.append(stripped.strip('"'))

In [ ]:
n_max = 10

In [ ]:
cluster_name_counts = Counter(cluster_names)

#Contagem do total de nós
node_count = len(cluster_names)

#DataFrame com todos os clusters, ordenados pelo número de nós
df_all_clusters = pd.DataFrame(
    sorted(cluster_name_counts.items(), key=lambda x: (-x[1], x[0])),
    columns=["ClusterName", "Count"]
)

#Configurações de exibição
summary_all = pd.DataFrame([["Total Nodes", node_count]], columns=["ClusterName", "Count"])
df_resultado = pd.concat([summary_all, df_all_clusters], ignore_index=True)

df_top10 = pd.concat([summary_all, df_all_clusters.head(n_max)], ignore_index=True)

data_table.DataTable(df_top10, num_rows_per_page=100)

In [ ]:
#Identificação dos índices dos nós de interesse
NEW_nodes = [i for i, cname in enumerate(cluster_names) if cname.startswith("A")]

In [ ]:
#Mapeamento antigo índice → novo índice
index_mapping = {old: new for new, old in enumerate(NEW_nodes)}

#Filtro das arestas internas do Cluster
if len(parts) == 2:
  NEW_edges = [(index_mapping[u], index_mapping[v]) for u, v in edges if u in index_mapping and v in index_mapping]
elif len(parts) == 3:
  NEW_edges = [
    (index_mapping[u], index_mapping[v], w)
    for u, v, w in edges
    if u in index_mapping and v in index_mapping
]

#Dados filtrados
NEW_titles = [titles[i] for i in NEW_nodes]
NEW_abstracts = [abstracts[i] for i in NEW_nodes]
NEW_wos_ids = [wos_ids[i] for i in NEW_nodes]

In [ ]:
#Caminho para salvar a sub-rede - Altere o nome se necessário
output_path = "/content/"

#Inserção dos dados na sub-rede
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(f"#vertices {len(NEW_nodes)}\n")
    for label in NEW_wos_ids:
        f.write(f'"{label}"\n')

    f.write("#edges nonweighted undirected\n")

    if len(parts) == 2:
      for u, v in NEW_edges:
          f.write(f"{u} {v}\n")
    elif len(parts) == 3:
      for u, v, w in NEW_edges:
          f.write(f"{u} {v} {w}\n")

    f.write('#v "abstract" s\n')
    for abs_txt in NEW_abstracts:
        f.write(f'"{abs_txt}"\n')

    f.write('#v "title" s\n')
    for title in NEW_titles:
        f.write(f'"{title}"\n')

print(f"Arquivo .xnet da sub-rede gerada em: {output_path}")
print(f'Número de nós da sub-rede: {len(NEW_nodes)}')
